# hplc-py 入門｜10 分鐘跑出你的第一張層析圖

這份 notebook **完全自給自足**：不用下載任何檔案，從上到下按 ▶ 執行就好。

搭配課程網站 `hplc-py-course.html` 使用。

---
### 怎麼用
1. 在 Colab 開啟這個檔案（檔案 → 上傳筆記本）
2. 每一格左邊的 ▶ 按下去，由上往下跑
3. 看到圖就成功了

> 不需要先會 Python。前三格只要按執行。


## 步驟 1｜安裝套件

第一次執行大約 30 秒。跑完可能會要求「重新啟動工作階段」，照做即可。


In [ ]:
!pip install --quiet hplc-py

import importlib.metadata as md
print('hplc-py 版本：', md.version('hplc-py'))


## 步驟 2｜準備範例資料

這裡直接用程式**生成**一張模擬層析圖，所以不用下載檔案。

模擬的是一個飲料樣品，含 6 個成分。因為是模擬的，我們**知道正確答案**，
待會可以檢查 hplc-py 有沒有算對——真實資料沒有這個好處。


In [ ]:
import numpy as np, pandas as pd
from scipy.special import erf

# (保留時間, scale, skew, 真實面積, 名稱)
TRUTH = [(4.20, 0.12, 1.5,  95.0, 'sucrose'),
         (5.60, 0.14, 2.0, 150.0, 'glucose'),
         (5.95, 0.15, 2.0,  58.0, 'fructose'),      # 和 glucose 重疊成肩峰
         (9.10, 0.18, 3.0, 180.0, 'citric acid'),
         (12.80, 0.20, 1.0,  72.0, 'benzoic acid'),
         (15.40, 0.22, 0.5, 120.0, 'caffeine')]

def skew_normal(t, tau, sigma, alpha, A):
    """論文 Eq.1 的峰形；A 就是這個峰的面積。"""
    z = (t - tau) / sigma
    return (A / np.sqrt(2*np.pi*sigma**2)) * np.exp(-0.5*z**2) * (1 + erf(alpha*z/np.sqrt(2)))

rng  = np.random.default_rng(20260730)      # 固定種子 -> 每次結果一樣
t    = np.arange(0.0, 20.0 + 1e-9, 0.01)    # 0-20 分鐘，每 0.01 分鐘一點
base = 1.5 + 0.35*t + 0.012*t**2            # 基線漂移（梯度沖提造成）
sig  = base + sum(skew_normal(t, *p[:4]) for p in TRUTH) + rng.normal(0, 0.8, t.size)

df = pd.DataFrame({'time': np.round(t, 3), 'signal': np.round(sig, 4)})
df.to_csv('demo_chromatogram.csv', index=False)
df.head()


> **如果你是在本機執行**，而且已經從課程網站下載了 `demo_chromatogram.csv`，
> 可以跳過上面那格，改用這一行讀檔——內容完全一樣：
>
> ```python
> df = pd.read_csv('demo_chromatogram.csv')
> ```
>
> 課程網站另外提供 `demo_ground_truth.csv`（6 個峰的正確答案），
> 內容就是上面那個 `TRUTH` 清單，後面幾章會用到。


## 步驟 3｜先看圖，再談分析

任何層析資料的第一件事都是**把它畫出來**。不要跳過這一步。


In [ ]:
import matplotlib.pyplot as plt

# 註：座標軸用英文，因為 Colab 預設沒有中文字型，中文會變成空白方框
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df['time'], df['signal'], lw=0.9, color='#10204b')
ax.set_xlabel('Retention time (min)')
ax.set_ylabel('Signal (mAU)')
ax.set_title('Demo chromatogram')
plt.show()

print('資料點數：', len(df))
print('取樣間隔：', round(df['time'].diff().median(), 4), 'min')
print('時間是否遞增：', df['time'].is_monotonic_increasing)


**看圖時問自己三件事：**

1. 有幾個峰？（數數看）
2. 基線是平的還是斜的？（這裡明顯往上斜）
3. 有沒有哪兩個峰黏在一起？（看 6 分鐘附近）


## 步驟 4｜用「預設參數」擬合——然後看它失敗

這一格刻意用預設參數。請注意最後印出的 `status`。


In [ ]:
from hplc.quant import Chromatogram

chrom_default = Chromatogram(df)
peaks_default = chrom_default.fit_peaks(verbose=False)      # 全部用預設
scores_default = chrom_default.assess_fit(verbose=False)

print('找到峰數：', len(peaks_default))
print()
print(scores_default[['window_type', 'time_start', 'time_end',
                      'reconstruction_score', 'status']].to_string(index=False))


### 發生什麼事？

6 個峰都找到了，參數也很接近正確答案——但 `status` 是 **invalid**，
而且整段 1–18 分鐘被當成**一個**峰窗。

原因是 `approx_peak_width` 的預設值是 **5 分鐘**，
但這份資料的峰寬只有大約 **0.5 分鐘**。

hplc-py 用這個值去估基線（SNIP 演算法）：
你告訴它「訊號大概 5 分鐘寬」，它就會把所有比 5 分鐘窄的東西當成訊號、
把基線估得太低，於是峰窗全部黏在一起。

> **這是整份教材最重要的一課：參數要由你的資料決定，不是沿用預設值。**


## 步驟 5｜給對峰寬，再跑一次


In [ ]:
chrom = Chromatogram(df)
peaks = chrom.fit_peaks(approx_peak_width=0.5, verbose=False)   # <-- 唯一的改動
scores = chrom.assess_fit(verbose=False)

print(scores[['window_type', 'time_start', 'time_end',
              'reconstruction_score', 'status']].to_string(index=False))


### 讀這張表

現在峰窗分成 5 段，`window_type == 'peak'` 的列 `reconstruction_score` 都很接近 1，
狀態是 **valid**。

`reconstruction_score`（課程裡叫 R）是「推論面積 ÷ 觀測面積」，
**不是 R²**。R = 1 代表面積對得起來。

> **不要被 `interpeak` 那幾列嚇到。** 峰與峰之間的空白區訊號幾乎是 0，
> 分母很小會讓比值變得不穩定，所以這些列幾乎一定顯示 `needs review`。
> 這是正常的，**判斷擬合好壞請只看 `window_type == 'peak'` 的列**。
> `interpeak` 真正要看的是它有沒有藏著漏掉的峰（課程第 8 章的 Fano ratio）。


In [ ]:
# 只看峰窗——這才是判斷擬合品質的依據
peak_windows = scores[scores['window_type'] == 'peak']
print(peak_windows[['time_start','time_end','reconstruction_score','status']]
      .to_string(index=False))
print()
print('全部峰窗都通過了嗎？', (peak_windows['status'] == 'valid').all())


In [ ]:
# 看看擬合結果長什麼樣
chrom.show()
plt.show()


## 步驟 6｜對答案

因為是模擬資料，我們知道真實面積。來看 hplc-py 算得準不準。

注意：對應真實面積的欄位是 **`amplitude`**，不是 `area`（下一步解釋）。


In [ ]:
truth = pd.DataFrame(TRUTH, columns=['rt_true','scale_true','skew_true','area_true','compound'])

cmp = pd.DataFrame({
    'compound'  : truth['compound'],
    'rt_true'   : truth['rt_true'],
    'rt_fit'    : peaks['retention_time'].values,
    'area_true' : truth['area_true'],
    'area_fit'  : peaks['amplitude'].values,
})
cmp['誤差_%'] = (100*(cmp['area_fit']-cmp['area_true'])/cmp['area_true']).round(2)
cmp.round(3)


### 讀這張表

- 保留時間幾乎完全正確
- 大多數成分的面積誤差在 ±1 % 以內
- **fructose 誤差比較大**（約 −2 %）：它和 glucose 重疊
- **caffeine 誤差最大**（約 −6 %）：它的峰最寬、最不對稱

> 結論：**重疊和拖尾會讓定量變差。** 這不是 hplc-py 的 bug，
> 而是所有層析定量的共同限制——只是這裡我們有答案可以量化它。


## 步驟 7｜`area` 與 `amplitude` 的單位陷阱

這一格很短但很重要。


In [ ]:
dt = 0.01   # 取樣間隔（分鐘）
chk = pd.DataFrame({
    'area'      : peaks['area'],
    'amplitude' : peaks['amplitude'],
    'area*dt'   : peaks['area'] * dt,
})
print(chk.round(4).to_string(index=False))
print()
print('area*dt 等於 amplitude 嗎？',
      np.allclose(peaks['area']*dt, peaks['amplitude']))


`area` 的單位是「訊號 × **取樣點數**」，不是「訊號 × 分鐘」。
`amplitude` 才是以時間為單位的積分面積。

同一台儀器、同一個取樣率下，兩者只差一個固定倍數，做檢量線時會自動抵消。
**但如果換儀器或改取樣率，`area` 不能直接互相比較。**

報告數字前先確認你用的是哪一欄，以及它的單位。


## 動手試試

改改看，再重跑，觀察結果怎麼變：

1. 把步驟 5 的 `approx_peak_width` 改成 `0.2` 和 `1.0`，看 `status` 怎麼變。
2. 在步驟 2 把雜訊 `rng.normal(0, 0.8, ...)` 的 `0.8` 改成 `5`，
   最小的峰還找得到嗎？
3. 把 fructose 的保留時間從 `5.95` 改成 `5.70`（更靠近 glucose），
   hplc-py 還分得開嗎？面積誤差變多少？

第 3 題就是課程「重疊峰與 known_peaks」那一章要解決的問題。


---

### 參考

- Chure, G., & Cremer, J. (2024). hplc-py: A Python Utility For Rapid Quantification
  of Complex Chemical Chromatograms. *JOSS*, 9(94), 6270. https://doi.org/10.21105/joss.06270
- 官方文件 https://cremerlab.github.io/hplc-py/

本 notebook 於 hplc-py **0.2.8** 實測通過。
